# Initialize

In [2]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import json
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import time
from pprint import pprint
from ec3 import EC3Materials
from ec3 import EC3epds
from itertools import (combinations, permutations, product)
from datetime import datetime


token = os.environ['EC3_KEY']
ec3_epds = EC3epds(bearer_token=token, ssl_verify=False)


In [3]:
categories = ['gwp', 'uuids', 'declared_unit', 'density_kgm3', 'date_pulled', 'n_EPD_i', 'n_EPD_f', 'label']

plt.rc('text',usetex=False)
mpl.rc('text',usetex=False)
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
afont = {'fontname': 'Arial'}

In [4]:
# #initialize json with dumps
# material = 'blanket_m2rsi'
# mat_dict = dict.fromkeys(categories)
# upload_dict = dict.fromkeys([material])
# upload_dict[material] = mat_dict

# json_object = json.dumps(upload_dict, indent=4)
# with open('kde_dict.json', 'w') as write_file:
#     write_file.write(json_object)

# EC3 loading functions

In [5]:
#load the complete category tree used by EC3
with open('masterformat_ec3_map.json') as json_file:
    epd_masterformats = json.load(json_file)

In [6]:
pprint(epd_masterformats)

[{'id': 'cb0a37f098f547a58fffb297eebe333a',
  'masterformat': '00 00 00 No Masterformat Code',
  'name': 'AllMaterials'},
 {'id': 'f4b20196cf844933b4222809f51386fa',
  'masterformat': '32 10 00 Bases Ballasts and Paving',
  'name': 'Aggregates'},
 {'id': '484df282d43f4b0e855fad6b351ce006',
  'masterformat': '03 00 00 Concrete',
  'name': 'Concrete'},
 {'id': '95793235b70c4b89b30d6a881ab8aee2',
  'masterformat': '03 61 00 Cementitious Grouting',
  'name': 'CementGrout'},
 {'id': 'f2efeb8dd3ac464783977a5b9d627a1d',
  'masterformat': '00 00 00 No Masterformat Code',
  'name': 'OilPatch'},
 {'id': 'cca4044f110c43aab149b8387c28f749',
  'masterformat': '03 37 13 Shotcrete',
  'name': 'Shotcrete'},
 {'id': '934cc1a8563041408ef75aaf549697f3',
  'masterformat': '32 13 13 Concrete Paving',
  'name': 'ConcretePaving'},
 {'id': 'b03dba1dca5b49acb1a5aa4daab546b4',
  'masterformat': '03 30 00 Cast-in-Place Concrete',
  'name': 'ReadyMix'},
 {'id': '8cce29c179d249afb3f2e1b9c1bce877',
  'masterformat'

# Function definitions

## Unit definitions

In [7]:
#Initialize variables

#prefixes and suffixes
hi2 = r'$\mathrm{\mathsf{   ^2   }}$'
hi3 = r'$\mathrm{\mathsf{   ^3   }}$'
lo2 = r'$\mathrm{\mathsf{   _2   }}$'


#weight
gs = ['g', 'gs']
kgs = ['kg', 'kgs']
lbs = ['lb', 'lbs', 'pound', 'pounds']
tons = ['t', 'ton', 'short', 'shortton']
tonnes = ['metric', 'tonne', 'metricton']
weight_units = gs + kgs + lbs + tons + tonnes

#area
m2 = ['m2', 'm²', 'm^2']
ft2 = ['ft2', 'sqft', 'sf', 'ft²', 'ft^2']
area_units = m2 + ft2

#volume
ft3 = ['ft3', 'ft^3', 'cuft', 'ft³']
m3 = ['m3', 'm³', 'm^3']
yd3 = ['yd3', 'yd³', 'cy', 'yd^3']
volume_units = ft3 + m3 + yd3

#density
kg_m3 = ['kg/m³', 'kg/m3', 'kg/m^3']
kg_yd3 = ['kg/yd3', 'kg/yd³']
lb_ft3 = ['lbs/ft3', 'lb/ft3', 'lb/ft³', 'pcf']
kg_dm3 = ['kg/dm3']
t_m3 = ['t/m3', 't/m³']
density_units = kg_m3 + kg_yd3 + lb_ft3 + kg_dm3 + t_m3

#conversion factors
m2_ft2 = 0.092903
m3_ft3 = 0.0283168
m3_yd3 = 0.764555


################################################################################################################
################################################################################################################

def weight2kgs(qty, unit, prnt='y'):
    """
    INPUTS: (qty, unit, prnt='y')
    OUTPUTS: qty (in kg)
    """
    if unit in kgs:
        pass
        
    elif unit in gs:
        qty /= 1000
        
    elif unit in lbs:
        qty /= 2.20462
    
    elif unit in tons:
        qty *= 2000/2.20462
        
    elif unit in tonnes:
        qty *= 1000
        
    else:
        qty = None
        if prnt == 'y':
            print(f'{unit} is not being accounted for in the weight2kgs function')
        
    return qty

################################################################################################################
################################################################################################################

def area2m2(qty, unit, prnt='y'):
    """
    INPUTS: (qty, unit, prnt='y')
    OUTPUTS: qty (in m2)
    """
    if unit in m2:
        pass
    
    elif unit in ft2:
        qty *= m2_ft2
        
    else:
        qty = None
        if prnt == 'y':
            print(f'{unit} is not being accounted for in the area2m2 function')
        
    return qty

################################################################################################################
################################################################################################################

def vol2m3(qty, unit, prnt='y'):
    """
    INPUTS: (qty, unit, prnt='y')
    OUTPUTS: qty (in m3)
    """
    if unit in m3:
        pass
    
    elif unit in ft3:
        qty *= m3_ft3
        
    elif unit in yd3:
        qty *= 27*m3_ft3
        
    else:
        qty = None
        if prnt == 'y':
            print(f'{unit} is not being accounted for in the vol2m3 function')
    
    return qty

################################################################################################################
################################################################################################################

def density2kgm3(qty, unit, prnt='y'):
    """
    INPUTS: (qty, unit, prnt='y')
    OUTPUTS: qty (in m3)
    """
    if unit in kg_m3:
        pass
    
    elif unit in kg_yd3:
        qty *= 1/27/m3_ft3
        
    elif unit in lb_ft3:
        qty *= 1/m3_ft3/2.20462
        
    elif unit in kg_dm3:
        qty *= 1000
    
    elif unit in t_m3:
        qty*= 1000
        
    else:
        qty = None
        if prnt=='y':
            print(f'{unit} is not being accounted for in the density2kgm3 function')
        
    return qty

################################################################################################################
################################################################################################################

def str2valunit(string):
    """
    INPUTS: (string)
    OUTPUTS: value, unit
    """
    if type(string) != str:
        raise ValueError('Input for str2valunit must be a string')
    string = string.replace(' ','').lower()
    tf_list = [any([ele.isdigit(), ele=='.', ele=='-', ele=='+']) for ele in list(string)]
    e_loc = string.find('e')
    if e_loc != -1 and e_loc != len(string)-1 and tf_list[e_loc+1] == True: #in case the value is in the format 1234e-2
        tf_list[e_loc] = True
    
    x = tf_list.index(False)
    value = float(string[:x])
    unit = string[x:]
    return value, unit

## extract_plot(data, unit, mat, alpha=0.25, plot='yes')

In [8]:
def extract_plot(data, unit, mat, alpha=0.25, plot='yes'):
    df = pd.DataFrame(data=data);
    df.sort_values(by='gwp', inplace=True, ascending=False)
    
    if plot == 'yes' or plot == 'y':
        fig, ax = plt.subplots(1,1)
        fig.set_size_inches(3,0.5)
        sns.boxplot(df, orient='h', color='white', saturation=0, fliersize=0, ax=ax)
        sns.stripplot(df, orient='h', alpha=alpha, size=5, ax=ax)
        
        ax.set_title(f'{mat} GWP, n={len(df)}',fontsize=10);
        ax.tick_params(axis='x', labelsize=6)
        ax.set_yticks([])
        ax.set_box_aspect(1/8)
        ax.spines[['left', 'top', 'right']].set_visible(False)
        ax.set_xlim(0,)
    return df

## Extract by UUID - extract_gwp_kg_uuid(epd_dict, mat, alpha=0.25, plot='y', prnt='y')

In [9]:
def extract_gwp_kg_uuid(epd_dict, mat, alpha=0.25, plot='y', prnt='y'):
    gwp_kg=[]
    uuids = []
    for epd in epd_dict[mat]:
        uuid = epd['open_xpd_uuid']
        if uuid not in impact_dict:
            continue
        else:
            gwp = impact_dict[uuid]

        qtys = []
        units = []
        if 'declared_unit' in epd:
            qty, unit = str2valunit(epd['declared_unit'])
            qtys.append(qty)
            units.append(unit)
        if 'mass_per_declared_unit' in epd:
            qty, unit = str2valunit(epd['mass_per_declared_unit'])
            qtys.append(qty)
            units.append(unit)
        if 'category' in epd and 'declared_unit' in epd['category']:
            qty, unit = str2valunit(epd['category']['declared_unit'])
            qtys.append(qty)
            units.append(unit)
        

        tf = [ele in weight_units for ele in units]
        tf_alt = [ele in volume_units for ele in units]

        if True in tf:
            ind = tf.index(True)
        elif True in tf_alt and 'density' in epd:
            ind = tf_alt.index(True)
            dens, dens_unit = str2valunit(epd['density'])
            if dens_unit not in density_units or units[ind] not in volume_units:
                continue
            kgm3 = density2kgm3(dens, dens_unit, prnt='y')
            m3 = vol2m3(qtys[ind], units[ind], prnt='y')
            qtys[ind] = m3*kgm3
            units[ind] = 'kg'
        else:
            continue

        qty = weight2kgs(qtys[ind], units[ind], prnt=prnt)
        gwp_kg.append(gwp/qty)
        uuids.append(uuid)
            
    data = {'gwp': gwp_kg,
            'uuids': uuids,
           }
    df = extract_plot(data=data, unit='kg', mat=mat, alpha=alpha, plot=plot)        
    return df

In [10]:
def extract_gwp_m2_uuid(epd_dict, mat, alpha=0.25, plot='y', prnt='y'):
    gwp_m2=[]
    uuids = []
    for epd in epd_dict[mat]:
        uuid = epd['open_xpd_uuid']
        if uuid not in impact_dict:
            continue
        else:
            gwp = impact_dict[uuid]

        qtys = []
        units = []
        if 'declared_unit' in epd:
            qty, unit = str2valunit(epd['declared_unit'])
            qtys.append(qty)
            units.append(unit)
        if 'category' in epd and 'declared_unit' in epd['category']:
            qty, unit = str2valunit(epd['category']['declared_unit'])
            qtys.append(qty)
            units.append(unit)

        tf = [ele in area_units for ele in units]

        if True in tf:
            ind = tf.index(True)
        else:
            continue

        qty = area2m2(qtys[ind], units[ind], prnt=prnt)
        gwp_m2.append(gwp/qty)
        uuids.append(uuid)

    data = {'gwp': gwp_m2,
            'uuids': uuids,
           }
    df = extract_plot(data=data, unit='m2', mat=mat, alpha=alpha, plot=plot)        

    return df

In [11]:
def extract_gwp_m3_uuid(epd_dict, mat, alpha=0.25, plot='y', prnt='y'):
    gwp_m3=[]
    uuids = []
    for epd in epd_dict[mat]:
        uuid = epd['open_xpd_uuid']
        if uuid not in impact_dict:
            continue
        else:
            gwp = impact_dict[uuid]

        qtys = []
        units = []
        if 'declared_unit' in epd:
            qty, unit = str2valunit(epd['declared_unit'])
            qtys.append(qty)
            units.append(unit)
        if 'category' in epd and 'declared_unit' in epd['category']:
            qty, unit = str2valunit(epd['category']['declared_unit'])
            qtys.append(qty)
            units.append(unit)

        tf = [ele in volume_units for ele in units]
        tf_alt = [ele in weight_units for ele in units]

        if True in tf:
            ind = tf.index(True)
        elif True in tf_alt and 'density' in epd:
            ind = tf_alt.index(True)
            dens, dens_unit = str2valunit(epd['density'])
            if dens_unit not in density_units or units[ind] not in weight_units:
                continue
            kgm3 = density2kgm3(dens, dens_unit, prnt='y')
            kg = weight2kgs(qtys[ind], units[ind], prnt='y')
            qtys[ind] = kg/kgm3
            units[ind] = 'm3'
        
        else:
            continue

        qty = vol2m3(qtys[ind], units[ind], prnt=prnt)
        gwp_m3.append(gwp/qty)
        uuids.append(uuid)

    data = {'gwp': gwp_m3,
            'uuids': uuids,
           }
    df = extract_plot(data=data, unit='m3', mat=mat, alpha=alpha, plot=plot)
    return df

## return_densities(epds, prnt='y')

In [12]:
kgm3_pcf = 16.018463373960042
m3_yd3 = 1/1.30795
def return_densities(epds, prnt = 'y'):
    densities = []
    for epd in epds:
        if 'density' in epd:
            density, unit = str2valunit(epd['density'])
            density = density2kgm3(density, unit, prnt=prnt)
            densities.append(density)

            if unit not in density_units and prnt == 'y':
                print(f"{unit} is not being accounted for in the return_densities function for {epd['open_xpd_uuid']}")
    
    return densities

## find_keys(epds, sub='')

In [13]:
def find_keys(epds, sub=""):
    #look for sub string in all keys and nested dictionaries
    all_keys = []
    for epd in epds:
        keys = epd.keys()
        some_keys = [s for s in keys if sub in s]
        for i in some_keys:
            if i not in all_keys:
                all_keys.append(i)

        #look in nested dictionaries
        for key in keys:
            if type(epd[key]) == dict:
                subkeys = epd[key].keys()
                some_subkeys = [s for s in subkeys if sub in s]
                for i in some_subkeys:
                    subkey = f'{key}:{i}'
                    if subkey not in all_keys:
                        all_keys.append(subkey)

    all_keys.sort()
    return all_keys

## return_declared_units(epds)

In [14]:
def return_declared_units(epds):
    units = []
    col = 'declared_unit'
    for epd in epds:
        if col in epd:
            units.append(str2valunit(epd[col])[1])
        else:
            units.append('N/A')

    return pd.DataFrame(data=units, columns=[col]).value_counts(subset=col)


## get_kg_m2(epds, prnt='y')

In [15]:
def get_kg_m2(epds, prnt='y'):
    kg_m2rsi = []
    for epd in epds:
        if 'declared_unit' in epd and 'mass_per_declared_unit' in epd:
            qty, qty_unit = str2valunit(epd['declared_unit'])
            mass, mass_unit = str2valunit(epd['mass_per_declared_unit'])
            
            mass = weight2kgs(mass, mass_unit, prnt=prnt)
            qty = area2m2(qty, qty_unit, prnt=prnt)
            if mass != None and qty != None:
                kg_m2rsi.append(mass/qty)

    kg_m2rsi.sort()
    return kg_m2rsi

# Pull A1-A3 UUIDs from excel sheet (20sec)

In [16]:
%%time
df_impacts = pd.read_csv('../../Dissertation/Manuscript_2023_KDE/Impacts_in_EPDs.csv')
df_impacts = df_impacts[df_impacts['i.name'] == 'gwp']
df_impacts = df_impacts[['e.open_xpd_uuid', 'i.name',
                        'i.a1_mean', 'i.a1_unit',
                        'i.a2_mean', 'i.a2_unit',
                        'i.a3_mean', 'i.a3_unit',
                        'i.a1a2a3_mean', 'i.a1a2a3_unit',
                       ]]
df_impacts['e.open_xpd_uuid'] = df_impacts['e.open_xpd_uuid'].str.lower()

#get units
units = []
stages = ['a1', 'a2', 'a3', 'a1a2a3']
for stage in stages:
    for ele in df_impacts[f'i.{stage}_unit'].unique():
        if ele not in units:
            units.append(ele)
units.remove(np.nan)
units.sort()

#convert values to kgCO2e. Introduce factor to be multiplied by value (previous value * factor = kg CO2e)
unitdict = {
    '1kgCO2e': 1,
    'gCO2e': 0.001,
    'kg CO2 eq.': 1,
    'kg CO2e': 1,
    'kgCO2': 1,
    'kgCO2e': 1,
    'kgCO2eq': 1,
    'tCO2e': 1000,
}


#check that all units from df_impacts are in the unitdict above
for ele in units:
    if ele not in unitdict:
        raise ValueError(f"'{ele}' not in a1a2a3_unitdict")

for stage in stages:
    df_impacts[f'{stage}_factor'] = df_impacts[f'i.{stage}_unit'].map(unitdict)
    df_impacts[f'{stage}'] = df_impacts[f'i.{stage}_mean'] * df_impacts[f'{stage}_factor']
    df_impacts.drop(columns=[f'i.{stage}_mean', f'i.{stage}_unit', f'{stage}_factor'], axis=1, inplace=True)


    
#get rid of rows that are the same
df_impacts.drop_duplicates(inplace=True, ignore_index=True)

# if there's a row with no NaNs for that uuid, delete the other rows
uuid_multiples = df_impacts['e.open_xpd_uuid'].value_counts().loc[lambda x : x>1].index
rel_stds = []
index2del = []
for uuid in uuid_multiples:
    df_test = df_impacts.loc[df_impacts['e.open_xpd_uuid'] == uuid]
    if len(df_test.dropna()) > 0:
        index2del += list(pd.isnull(df_test.T).any()[lambda x : x==True].index)

df_impacts = df_impacts.drop(index=index2del)

#pick the first iteration when there are multiple rows for the same uuid
uuid_multiples = df_impacts['e.open_xpd_uuid'].value_counts().loc[lambda x : x>1].index
index2del = []
for uuid in uuid_multiples:
    index2del += list(df_impacts.loc[df_impacts['e.open_xpd_uuid'] == uuid].index[1:])
    
df_impacts = df_impacts.drop(index=index2del)
impact_dict = dict(zip(df_impacts['e.open_xpd_uuid'], df_impacts['a1a2a3']))

CPU times: user 16.9 s, sys: 512 ms, total: 17.4 s
Wall time: 17.5 s


In [17]:
#

# _
# _
# _ 
# Read EPDs from EC3 according to ec3 parameters (2h, 8m with concrete, otherwise 15m)

In [18]:
%%time

#if list,   ec3_epds.get_epds(return_all=True, {'category': i}) for i in list
#elif dict, ec3_epds.get_epds(return_all=True, dict)
ec3_ids = {
    # 'concrete': ['b03dba1dca5b49acb1a5aa4daab546b4'], # 'masterformat': '03 30 00 Cast-in-Place Concrete', 'name': 'ReadyMix'
    'stone': ['f4b20196cf844933b4222809f51386fa'],
    'fgbatt': {'insulating_material': 'Fiberglass'}, #insulating_material options: Mineral Wool, Cellulose, Fiberglass, XPS, GPS, Polyisocyanurate, Expanded Polyethylene, EPS, Wood Fiber, Other
    'glass': ['ade3ad3405124279955e7d3085f59383'], #   'masterformat': '08 81 23 Exterior Glass Glazing', 'name': 'InsulatingGlazingUnits'
    'gypsum': [
        '0d1d4544c70f4a82938c79d88f86ec5d', #'masterformat': '09 29 00 Gypsum Board', 'name': 'Gypsum'},
        '4063336edfe749b480a9f932d4109fe6', #'masterformat': '06 16 43 Gypsum Sheathing', 'name': 'GypsumSheathingBoard'},
    ],
    'plywood': {'fabrication':'Plywood',},
    'roofmembranes':[
        '032b9e6f8fc24d30a21ad62a639678dd', #  'masterformat': '07 46 24 Wood Shingle and Shake Siding','name': 'ShingleAndShakeSiding'},
        'd4f59a8702314e84b683dde20c2a8a47', #  'masterformat': '07 50 00 Membrane Roofing', 'name': 'MembraneRoofing'},
        'b9ec3e557d8d473fa4f8aa2af05b9cd4', #  'masterformat': '07 50 00 Membrane Roofing', 'name': 'SinglePlyEPDM'},
        '34d34329c1964de4a9b67437ef00c4ea', #  'masterformat': '07 50 00 Membrane Roofing', name': 'SinglePlyKEE'},
        'e95e0d13de844101beb364b47af73d45', #  'masterformat': '07 50 00 Membrane Roofing', 'name': 'SinglePlyPolyurethane'},
        'a9fd6c7eca5a4d5382ece6423f0326ec', #  'masterformat': '07 50 00 Membrane Roofing', 'name': 'BituminousRoofing'},
        '1b02f218bdaf4f31aae48192221dc7e5', #  'masterformat': '07 50 00 Membrane Roofing', 'name': 'SinglePlyOther'},
        'db8d5fc34f304dd69cf4829d456d3c69', #  'masterformat': '07 50 00 Membrane Roofing', 'name': 'SinglePlyTPO'},
        'eea8edb2c0f84d77abfab5a6241cbae1', #  'masterformat': '07 50 00 Membrane Roofing', 'name': 'SinglePlyPVC'},
        '7bbff43e19124317a307ede6e0179b4c', #  'masterformat': '07 30 00 Steep Slope Roofing', 'name': 'SteepSlopeRoofing'},
        '4ecbbb1ffe224f1b960fdaaf41c503a4', #  'masterformat': '07 22 17 Low Slope Roofing Cover Board', 'name': 'RoofCoverBoards'},
        'e0e179593b504a1b986e849cade3ff15', #  'masterformat': '07 40 00 Roofing and Siding Panels', 'name': 'Cladding'},
        '154ea9e0d9964ddc873cacd7c1968327', #  'masterformat': '07 42 00 Wall Panels', 'name': 'WallPanels'},
        'b9bd97e8c55b49288eb9d672882f72a8', #  'masterformat': '07 41 00 Roof Panels', 'name': 'RoofPanels'},
        '335621cd7eec42e89c2b76bb2a05e4fa', #  'masterformat': '07 41 00 Roof Panels', 'name': 'InsulatedRoofPanels'},
        '760b95e84f6d499db89914f59ecf0902', #  'masterformat': '08 45 00 Translucent Wall and Roof Assemblies', 'name': 'TranslucentWallAndRoofAssemblies'},
    ],
    'steelrebar': ['4e82b07fff2e40aeb1b72b695e9ac279'], #'masterformat': '03 21 00 Reinforcement Bars', 'name': 'RebarSteel'},
    'wood': [
        'fd14efc8874c4a55ac30d84a5612feb1', #'masterformat': '06 10 00 Rough Carpentry', 'name': 'Timber'},
        '0af2a872f37d4ffcbaff4485153206c4', #'masterformat': '06 10 00 Rough Carpentry', 'name': 'Unfinished'},
        'bdc042c5cca3498bbf6bf039d8a97261', #'masterformat': '06 11 00 Wood Framing', 'name': 'DimensionLumber'},
        '64d9985c1a5240c1b2a0b2b36b285a1b', #'masterformat': '06 15 00 Wood Decking', 'name': 'WoodDecking'},
        'ce2498926db44d0d9177c7bfdf3520c0', #'masterformat': '06 11 00 Wood Framing', 'name': 'WoodFraming'},
        'e4aa9c1808ad41b6944db88e51d877ba', #'masterformat': '06 16 00 Sheathing', 'name': 'SheathingPanels'},
        '42897282978b438598c1d7315dff7036', #'masterformat': '06 13 00 Heavy Timber Construction', 'name': 'HeavyTimber'},
        'ef49fef3f7c240e18d6f32dfd9583bfc', #'masterformat': '06 18 00 Glued-Laminated Construction', 'name': 'MassTimber'},
        '9717c6b40208445fbea4c7c5fbfb9557', #'masterformat': '06 17 00 Shop-Fabricated Structural Wood', 'name': 'CompositeLumber'},
        '6495a039f98f42bdbec921121d48dbc4', #'masterformat': '06 10 00 Rough Carpentry', 'name': 'PrefabricatedWood'},
        '23374ad8268e4e7d89b9fd3316f1bbcb', #'masterformat': '06 17 53 Shop-Fabricated Wood Trusses', 'name': 'PrefabricatedWoodTruss'},
    ],
    'xps': {'insulating_material':'XPS'}, #insulating_material options: Mineral Wool, Cellulose, Fiberglass, XPS, GPS, Polyisocyanurate, Expanded Polyethylene, EPS, Wood Fiber, Other



##############################################################################
# NON DOE MATERIALS
#############################################################################
    'blanket': ['53a5d5bee64545f1bdd60e102a4a6ddf'],
    'brick': ['41ff8f44d89c4cdbbcdf2671a35286f5'],
    'carpet': ['b43b707159f044bc90352dcb2b2778e3'],
    'cellulose': {'insulating_material':'Cellulose'}, #Mineral Wool, Cellulose, Fiberglass, XPS, GPS, Polyisocyanurate, Expanded Polyethylene, EPS, Wood Fiber, Other
    'tile': ['322f5fe92dc14206868213d0f800fe80'],
    'eps': {'insulating_material':'EPS'},
    'siding': [
        '088ba9fc78a24ab1b5b1f93c0c44a96a', #'09 28 13 Cementitious Backing Board',
        'a465ce986abd45bfa00335e9d007216a',#'09 28 00 Backing Boards and Underlayments',
        '0d1d4544c70f4a82938c79d88f86ec5d',#'09 29 00 Gypsum Board',
        '4f5c4543615e4e5790eaf5415c9a2bfc',#'09 22 00 Supports for Plaster and Gypsum Board',
        'f072c0f08eae4de598c7e3cae88c1e8a',#'06 00 00 Wood, Plastics & Composites',
        '4ecbbb1ffe224f1b960fdaaf41c503a4',#'07 22 17 Low Slope Roofing Cover Board',
        '2ba88414ca7344f6bf4140ca1bd536de',#'06 16 00 Sheathing',
        '4063336edfe749b480a9f932d4109fe6',#'06 16 43 Gypsum Sheathing',
        'c56db9d02dfe4fb2bb8cf0eb5f35a446',#'06 16 63 Cementitious Sheathing',
        'e0e179593b504a1b986e849cade3ff15',#'07 40 00 Roofing and Siding Panels',
        'e4aa9c1808ad41b6944db88e51d877ba',#'06 16 00 Sheathing',
        '154ea9e0d9964ddc873cacd7c1968327',#'07 42 00 Wall Panels',
        'b9bd97e8c55b49288eb9d672882f72a8',#'07 41 00 Roof Panels',
        '335621cd7eec42e89c2b76bb2a05e4fa',#'07 41 00 Roof Panels',
        '8eb6a511c7d94f4f89e673b6e9badd3c',#'07 42 00 Wall Panels',
        '47413201bd134c84a5d342c85d0a086b',#'07 46 00 Siding'
        '65d35d06cae846a8867b581d4a8b6fd9',#'07 46 29 Plywood Siding',
        'bd12ff6c23584255aa6b4600c58c3768',#'07 46 43 Composition Siding',
        '58c9924ec5c648fa9096af1f460c6911',#'07 46 00 Siding',
        '700f2e5933e84c4fb3f56a6e9b80cb93',#'07 46 23 Wood Siding',
        '032b9e6f8fc24d30a21ad62a639678dd',#'07 46 24 Wood Shingle and Shake Siding',
        'e928f568d5a547f5b8ae7f58f44fc232',#'07 46 00 Siding',
        'e4005886c2a34303af17a33a09d7e1f3',#'07 46 21 Zinc Siding',
        'b2de25405a374b92b41f5c88c6aac88d',#'07 46 19 Steel Siding',
        'd326742d916b42868ca1d052d5f68249',#'07 46 16 Aluminum Siding',
        '7c982d586f0549cc9fe260bdb1634eb5',#'07 46 00 Siding',
        'b6cc461ee3484d9d958eba6bdb2b18a5',#'07 46 00 Siding',
        'ad56c06d77e64ca6881bf08d48f31919',#'07 46 46 Fiber-Cement Siding',
    ],
    'metalsheet': ['934cc56fb68a4757a0349ab293b55994'],
    'mineralwool': {'insulating_material':'Mineral Wool'}, #Mineral Wool, Cellulose, Fiberglass, XPS, GPS, Polyisocyanurate, Expanded Polyethylene, EPS, Wood Fiber, Other
    'osb': {'fabrication':'OSB'},
    'insulation': [
        'bf1c8882d7784db4b10d9d5698b8b5cc', #07 21 00 thermal insulation, Insulation
        '56f3c898f94b459eb18feadeb792ab88', #07 21 13 board insulation, BoardInsulation
        '53a5d5bee64545f1bdd60e102a4a6ddf', #07 21 16 blanket insulation, BlanketInsulation
        '4a04f3715b144b09ab4f59e18aca1122', #07 21 16 blanket insulation, BlanketFacing
        '712c2265bfe248e5aa58ad17bb3c92ea', #07 21 19 foamed-in-place insulation, FoamedInPlace
        '6fd418c8ff92415c833e6327638d8482', #07 21 26 blown insulation, BlownInsulation
    ],
    'sprayfoam': ['712c2265bfe248e5aa58ad17bb3c92ea'],
    'steelcoldformed': ['df6ecf896ab5433b8d180099b3c43940'],
    'steeldeck': ['05d49f750eed4648b7f72e3bbfc8b8e4'], #05 31 00 steel decking
    'steelstructural': [
        '820e86326d2945fb9322246fd4690fc8', #05 12 00 structural steel framing, StructuralSteel
        'de95ab7d6ab5488bb87d20177f942d2a', #05 12 00 structural steel framing, HotRolled
        'e96e409053534524b18ca57b992bb328', #05 12 00 structural steel framing, Hollow
        '3c2208e79d954872af198013ae046a26', #05 12 00 structrual steel framing, PlateSteel
    ],
    
}




#initialize epd_dict
epd_dict = dict.fromkeys(ec3_ids.keys())

# run through ec3_ids and get all EPDs from EC3
ec3_epds = EC3epds(bearer_token=token, ssl_verify=False)
ec3_materials = EC3Materials(bearer_token=token, ssl_verify=False)

for mat in ec3_ids:
    params = ec3_ids[mat]
    epds = []
    
    if type(params)==list: #masterformat ids
        for i in params:
            print(i)
            param_dict = {'category': i}
            before = len(epds)
            epds += ec3_epds.get_epds(return_all=True, params=param_dict)
            # epds += ec3_materials.get_materials(return_all=True, params=param_dict) #takes WAY too long. Stopped at 'db8d5fc34f304dd69cf4829d456d3c69' for a while
            time.sleep(2.5)
    elif type(params)==dict: #param dict
        print(params)
        epds = ec3_epds.get_epds(return_all=True, params=params)
        # epds = ec3_materials.get_materials(return_all=True, params=params)
        time.sleep(2.5)
    else:
        raise ValueError('Make sure the data type in ec3_ids is either a dictionary (param dict) or list (of masterformat ids)')
    
    epd_dict[mat] = epds

epd_dict = dict(sorted(epd_dict.items())) #sort dictionary by keys

f4b20196cf844933b4222809f51386fa
{'insulating_material': 'Fiberglass'}
ade3ad3405124279955e7d3085f59383
0d1d4544c70f4a82938c79d88f86ec5d
4063336edfe749b480a9f932d4109fe6
{'fabrication': 'Plywood'}
032b9e6f8fc24d30a21ad62a639678dd
d4f59a8702314e84b683dde20c2a8a47
b9ec3e557d8d473fa4f8aa2af05b9cd4
34d34329c1964de4a9b67437ef00c4ea
e95e0d13de844101beb364b47af73d45
a9fd6c7eca5a4d5382ece6423f0326ec
1b02f218bdaf4f31aae48192221dc7e5
db8d5fc34f304dd69cf4829d456d3c69
eea8edb2c0f84d77abfab5a6241cbae1
7bbff43e19124317a307ede6e0179b4c
4ecbbb1ffe224f1b960fdaaf41c503a4
e0e179593b504a1b986e849cade3ff15
154ea9e0d9964ddc873cacd7c1968327
b9bd97e8c55b49288eb9d672882f72a8
335621cd7eec42e89c2b76bb2a05e4fa
760b95e84f6d499db89914f59ecf0902
4e82b07fff2e40aeb1b72b695e9ac279
fd14efc8874c4a55ac30d84a5612feb1
0af2a872f37d4ffcbaff4485153206c4
bdc042c5cca3498bbf6bf039d8a97261
64d9985c1a5240c1b2a0b2b36b285a1b
ce2498926db44d0d9177c7bfdf3520c0
e4aa9c1808ad41b6944db88e51d877ba
42897282978b438598c1d7315dff7036
ef49fef3f7c

# EPDs derived from those in EC3

In [19]:
%%time
#takes 10 minutes
#read json file
with open('epd_dict.json', 'r') as read_file:
    epd_dict_temp = json.load(read_file)

epd_dict['concrete'] = epd_dict_temp['concrete_m3']['epds'].copy()

CPU times: user 41.1 s, sys: 17.8 s, total: 58.9 s
Wall time: 1min 2s


In [50]:
%%time
# Materials derived from the categories already listed

# concreteTX_epds from concrete_epds
epd_dict['concreteTX'] = [epd for epd in epd_dict['concrete'].copy() if 'tx' in epd['plant_geography'][0].lower()]

# concreteTX4ksi_epds from concreteTX_epds
# concreteTX3ksi_epds from concreteTX_epds
epd_dict['concreteTX4ksi'] = []

epd_dict['concreteTX3ksi'] = []

for epd in epd_dict['concreteTX'].copy():
    if 'concrete_compressive_strength_28d' in epd:
        fc, unit = str2valunit(epd['concrete_compressive_strength_28d'])

        #+/-10% for 4ksi concrete = 27.58 MPa
        if unit == 'mpa' and fc < 30.34 and fc > 24.82:
            epd_dict['concreteTX4ksi'].append(epd)
        elif unit == 'psi' and fc < 4400 and fc > 3600:
            epd_dict['concreteTX4ksi'].append(epd)

        #+/-10% for 3ksi concrete = 20.68 MPa
        elif unit == 'mpa' and fc < 22.75 and fc > 18.62:
            epd_dict['concreteTX3ksi'].append(epd)
        elif unit == 'psi' and fc < 3300 and fc > 2700:
            epd_dict['concreteTX3ksi'].append(epd)
        else:
            continue

# concrete4ksi from concrete_epds
epd_dict['concrete4ksi'] = []
epd_dict['concrete3ksi'] = []

for epd in epd_dict['concrete'].copy():
    if 'concrete_compressive_strength_28d' in epd:
        fc, unit = str2valunit(epd['concrete_compressive_strength_28d'])

        #+/-10% for 4ksi concrete = 27.58 MPa
        if unit == 'mpa' and fc < 30.34 and fc > 24.82:
            epd_dict['concrete4ksi'].append(epd)
        elif unit == 'psi' and fc < 4400 and fc > 3600:
            epd_dict['concrete4ksi'].append(epd)
        
        #+/-10% for 3ksi concrete = 20.68 MPa
        elif unit == 'mpa' and fc < 22.75 and fc > 18.62:
            epd_dict['concrete3ksi'].append(epd)
        elif unit == 'psi' and fc < 3300 and fc > 2700:
            epd_dict['concrete3ksi'].append(epd)
        else:
            continue



# dimlumber from wood_epds
mat = 'dimlumber'
epd_dict[mat] = []
epd_dict[mat] = [epd for epd in epd_dict['wood'].copy() if 'description' in epd and 'dim' in epd['description'].lower()]
epd_dict[mat] = [epd for epd in epd_dict[mat] if 'varmebehandlet' not in epd['description'].lower()] #varmebehandlet is norwegian for heat-treated
epd_dict[mat] = [epd for epd in epd_dict[mat] if 'lvl' not in epd['description'].lower()]
epd_dict[mat] = [epd for epd in epd_dict[mat] if 'clt' not in epd['description'].lower()]
epd_dict[mat] = [epd for epd in epd_dict[mat] if 'cross laminated timber' not in epd['description'].lower()]
epd_dict[mat] = [epd for epd in epd_dict[mat] if 'engineered' not in epd['description'].lower()]
epd_dict[mat] = [epd for epd in epd_dict[mat] if 'glulam' not in epd['description'].lower()]
epd_dict[mat] = [epd for epd in epd_dict[mat] if 'metsä' not in epd['description'].lower()] #engineered panel product
epd_dict[mat] = [epd for epd in epd_dict[mat] if 'component' not in epd['description'].lower()] #fasteners

for epd in epd_dict[mat]:
    if 'dillard, or' in epd['description'].lower():
        epd['plant_geography'] = ['US-OR']



# shingles from roofmembranes_epds
mat = 'shingles'
epd_dict[mat] = []
for epd in epd_dict['roofmembranes'].copy():
    if 'description' in epd and 'shingle' in epd['description']:
        epd_dict[mat].append(epd)


# mat = 'shingles3psf_kg'
# epd_dict[mat] = [epd for epd in epd_dict['shingles'].copy() if 'mass_per_declared_unit' in epd]
# for epd in epd_dict[mat]:
#     epd['declared_unit'] = epd['mass_per_declared_unit']
#     if 'portland, or' in epd['description'].lower():
#         epd['plant_geography'] = ['US-OR']
        

#gypsum_kg
# mat = 'gypsum_kg'
# epd_dict[mat] = [epd for epd in epd_dict['gypsum'].copy() if 'mass_per_declared_unit' in epd]
# for epd in epd_dict[mat]:
#     epd['declared_unit'] = epd['mass_per_declared_unit']
    
    
    
##############################################################################
# NON DOE MATERIALS
#############################################################################
        

epd_dict['clay'] = [epd for epd in epd_dict['tile'].copy() if 'name' in epd and 'ceramic' in epd['name'].lower()]

epd_dict['clt'] = [epd for epd in epd_dict['wood'].copy() if 'description' in epd]
epd_dict['clt'] = [epd for epd in epd_dict['clt'] if 'clt' in epd['description'].lower() or 'cross' in epd['description'].lower()]

epd_dict['glulam'] = [epd for epd in epd_dict['wood'].copy() if 'description' in epd and 'glulam' in epd['description'].lower()]

epd_dict['gypsumlight'] = [epd for epd in epd_dict['gypsum'].copy() if 'description' in epd and 'lightw' in epd['description'].lower()]
epd_dict['gypsumlight'] = [epd for epd in epd_dict['gypsumlight'] if epd['open_xpd_uuid'] not in ['ec3e9973', 'ec3nm317', 'ec3rjbxm']]


epd_dict['hardieboard'] = [epd for epd in epd_dict['siding'].copy() if 'manufacturer' in epd and 'hardi' in str(epd['manufacturer']).lower()]
epd_dict['hardieboard'] = [epd for epd in epd_dict['hardieboard'] if 'plant_or_group' in epd and 'hardi' in str(epd['plant_or_group']).lower()]

epd_dict['rockwool'] = [epd for epd in epd_dict['insulation'].copy() if 'description' in epd and 'rockw' in epd['description'].lower()]

epd_dict = dict(sorted(epd_dict.items())) #sort dictionary by keys


CPU times: user 482 ms, sys: 783 ms, total: 1.27 s
Wall time: 1.3 s


# Filter out undesirable EPDs or put EPDs in the right units

In [51]:
%%time
epd_dict_filtered = epd_dict.copy()

#fgbatt
mat = 'fgbatt'
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'mineral' not in epd['description'].lower()]
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'stone' not in epd['description'].lower()]

#glass
mat = 'glass'
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'electrochromic' not in epd['description'].lower()]  
for i in range(len(epd_dict_filtered[mat])):
    if 'gwp_per_kg' in epd_dict_filtered[mat][i] and 'gwp' in epd_dict_filtered[mat][i]:
        gwp = str2valunit(epd_dict_filtered[mat][i]['gwp'])[0]
        gwp_per_kg = str2valunit(epd_dict_filtered[mat][i]['gwp_per_kg'])[0]
        weight_kg = gwp/gwp_per_kg
        epd_dict_filtered[mat][i]['declared_unit'] = f'{weight_kg} kg'

#plywood
mat = 'plywood'
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'gwp' not in epd or str2valunit(epd['gwp'])[0]>0] 
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'latvijas' not in epd['description'].lower()] 
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'flexible air sealing tape' not in epd['description'].lower()] 

#gypsum
mat = 'gypsum'
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'cement board' not in epd['description'].lower()]
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'glass-mat' not in epd['description'].lower()]
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'gypsum' in epd['description'].lower()]

for epd in epd_dict_filtered[mat]:
    if all([ele in epd for ele in ['density', 'gypsum_thickness', 'declared_unit']]):        
        density, density_unit = str2valunit(epd['density'])
        density_kgm3 = density2kgm3(density, density_unit)
        
        thickness, thickness_unit = str2valunit(epd['gypsum_thickness'])
        if thickness_unit.lower() == 'mm':
            thickness_m = thickness/1000
        elif thickness_unit.lower() == 'in':
            thickness_m = thickness/39.3701
        else:
            continue

        qty, qty_unit = str2valunit(epd['declared_unit'])
        qty_m2 = area2m2(qty, qty_unit, prnt='n')
        
        if None in [density_kgm3, thickness_m, qty_m2]:
            continue
        
        derived_mass_kg = density_kgm3*thickness_m*qty_m2
        
        epd['declared_unit'] = f'{derived_mass_kg} kg'
        
for i in range(len(epd_dict_filtered[mat])):
    uuid = epd_dict_filtered[mat][i]['open_xpd_uuid']
    if uuid in ['ec3w5bt3', 'ec36z3gf', 'ec3bzhx1', 'ec3sx590']:
        qty, unit = str2valunit(epd['declared_unit'])
        qty *= 10
        epd_dict_filtered[mat][i]['declared_unit'] = f'{qty} {unit}'

#stone
mat = 'stone'
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'light' not in epd['description'].lower()]
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'powder' not in epd['description'].lower()]
stone_density_kgm3 = 2650
for i in range(len(epd_dict_filtered[mat])):
    if 'declared_unit' in epd_dict_filtered[mat][i]:
        qty, unit = str2valunit(epd_dict_filtered[mat][i]['declared_unit'])
    else:
        continue
    
    if unit not in weight_units:
        continue
    weight_kg = weight2kgs(qty, unit)
    epd_dict_filtered[mat][i]['declared_unit'] = f'{weight_kg/stone_density_kgm3} m3'
    
#shingles
mat = 'shingles'
# assume 3 psf https://roofonline.com/weight-of-roofing-materials/?expand_article=1 
# assume 6.1mm thickness
kgm2_psf = 4.8824
shingle_psf = 3
shingle_kgm2 = shingle_psf*kgm2_psf

for i in range(len(epd_dict_filtered[mat])):
    if 'declared_unit' in epd_dict_filtered[mat][i]:
        qty, unit = str2valunit(epd_dict_filtered[mat][i]['declared_unit'])
    else:
        continue
    
    if unit not in area_units:
        continue
    area_m2 = area2m2(qty, unit)
    epd_dict_filtered[mat][i]['declared_unit'] = f'{area_m2*shingle_kgm2} kg'
        
        
##############################################################################
# NON DOE MATERIALS
#############################################################################
#blanket
epd_dict_filtered['blanket']  = [epd for epd in epd_dict_filtered['blanket'] if 'description' not in epd or 'climablock' not in epd['description'].lower()]

#brick
mat = 'brick'
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'composite material' not in epd['description'].lower()]
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'silicone-based' not in epd['description'].lower()]
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'ec3wafxe' not in epd['open_xpd_uuid'].lower()] #some weird block
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'ec3aufh2' not in epd['open_xpd_uuid'].lower()] #some weird block

#cellulose
mat = 'cellulose'
epd_dict_filtered[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'acoustic' not in epd['description'].lower()]
epd_dict[mat] = [epd for epd in epd_dict_filtered[mat] if 'description' not in epd or 'cement' not in epd['description'].lower()]

epd_dict_filtered['clt'] = [epd for epd in epd_dict_filtered['clt'] if 'description' not in epd or 'plywood' not in epd['description'].lower()]

epd_dict_filtered['mineralwool'] = [epd for epd in epd_dict_filtered['mineralwool'] if 'description' not in epd or 'panel' not in epd['description'].lower()]

epd_dict_filtered['sprayfoam'] = [epd for epd in epd_dict_filtered['sprayfoam'] if 'description' not in epd or 'xps' not in epd['description'].lower()]

epd_dict_filtered['steelstructural'] = [epd for epd in epd_dict_filtered['steelstructural'] if 'file_name' not in epd or 'salmon' not in epd['file_name'].lower()]

mat = 'osb'
uuid = 'ec3f3g6a'
for i, epd in enumerate(epd_dict_filtered[mat]):
    if epd['open_xpd_uuid'] == uuid:
        ind = i
epd_dict_filtered[mat][ind]['category']['declared_unit'] = epd_dict_filtered[mat][ind]['declared_unit']

mat = 'carpet'
uuids = ['ec329xb7', 'ec3mfenf']

for i, epd in enumerate(epd_dict_filtered[mat]):
    uuid = epd['open_xpd_uuid']
    if uuid in uuids:
        val = str2valunit(epd_dict_filtered[mat][i]['gwp'])[0]
        impact_dict[uuid] = str2valunit(epd_dict_filtered[mat][i]['gwp'])[0]


CPU times: user 28.5 ms, sys: 10.8 ms, total: 39.3 ms
Wall time: 38.3 ms


# Find declared_unit, extract GWP, plot strip/box

In [52]:
%%time
gwp_dict = dict.fromkeys(epd_dict_filtered.keys())
#determine most common type of unit for each material
unit_extraction_dict = {
    'weight': weight_units,
    'area': area_units,
    'volume': volume_units,
}

unit_count_series = pd.Series(index=unit_extraction_dict.keys(), dtype=float)

all_units = []
unaccounted_units = []

for unit_type in unit_extraction_dict:
    all_units += unit_extraction_dict[unit_type]
    
    
for mat in epd_dict_filtered:
    unit_count_dict = dict(return_declared_units(epd_dict_filtered[mat]))

    for unit_type in unit_extraction_dict:
        unit_count_series[unit_type] = 0
        for unit in unit_count_dict:
            if unit in unit_extraction_dict[unit_type]:
                unit_count_series[unit_type] += unit_count_dict[unit]

            if unit not in all_units and unit not in unaccounted_units:
                unaccounted_units.append(unit)


    unit_count_series.sort_values(ascending=False, inplace=True)    
    unit_count_series = unit_count_series.astype(int)

    most_declared_unit = unit_count_series.index[0]
    # if mat == 'glass':
    #     print(mat)
    #     print(unit_count_series)
    #     print('')
    if most_declared_unit == 'weight':
        df_gwp_by_uuid = extract_gwp_kg_uuid(epd_dict_filtered, mat=mat, plot='n')
        mat_declared_unit = 'kg'
    
    elif most_declared_unit == 'area':
        df_gwp_by_uuid = extract_gwp_m2_uuid(epd_dict_filtered, mat=mat, plot='n')
        mat_declared_unit = 'm2'
    
    elif most_declared_unit == 'volume':
        df_gwp_by_uuid = extract_gwp_m3_uuid(epd_dict_filtered, mat=mat, plot='n')
        mat_declared_unit = 'm3'
        
    else:
        raise ValueError(f'Something went wrong. "{most_declared_unit}" should be weight, area, or volume')
    
    gwp_dict[mat] = dict.fromkeys(categories)
    gwp_dict[mat]['gwp'] = list(df_gwp_by_uuid['gwp'])
    gwp_dict[mat]['uuids'] = list(df_gwp_by_uuid['uuids'])
    gwp_dict[mat]['declared_unit'] = mat_declared_unit
    
#update with units
#initialize dictionary that will change the units
add_material_unit_dict = {}

#create dictionary with old_key: new_key to add _unit
for mat in gwp_dict:
    loc_ = mat.find('_')
    declared_unit = gwp_dict[mat]['declared_unit']
    if loc_ == -1:
        new_key = f'{mat}_{declared_unit}'
    else:
        new_key = f'{mat[:loc_]}_{declared_unit}'
    
    add_material_unit_dict[mat] = new_key

#iterate through gwp_dict
for mat in add_material_unit_dict:
    gwp_dict[add_material_unit_dict[mat]] = gwp_dict.pop(mat)


gwp_dict = dict(sorted(gwp_dict.items())) #sort dictionary by keys
print('')
print(unaccounted_units)


['N/A', 'mm', 'm**2', 'item', 'g·mm', 'mi', 'm', 'kw', 'km', 'w', 'cm', 'l', 'mj', 'in', 'j', 'th', 'ft']
CPU times: user 898 ms, sys: 264 ms, total: 1.16 s
Wall time: 1.16 s


# (OPTIONAL): Check outliers on a case-by-case basis

Index(['ec379k32', 'ec3a65uz', 'ec3mua8g', 'ec33a9ys'], dtype='object')

In [55]:
#glass_kg - no issues with 2 values >3.5 ['ec3m65sd', 'ec3naa5z']
#concrete_m3 - no issues with 2 values ~2500 ['ec33pp3w', 'ec3k1ujk']
#concrete3ksi_m3 - no issues 14 values ~1000 ['ec3c399f', 'ec3c399f', 'ec3afs62', 'ec3afs62', 'ec3aag4a', 'ec3aag4a', 'ec3804nq', 'ec3804nq', 'ec3k3y28', 'ec3k3y28', 'ec35dgzr', 'ec35dgzr', 'ec3357e4', 'ec3357e4']

mat_unit = 'dimlumber_m3'
mat = mat_unit[:mat_unit.find('_')]
gwps = gwp_dict[mat_unit]['gwp']
index = gwp_dict[mat_unit]['uuids']
df_test = pd.DataFrame(data=gwps, index=index, columns=['gwp']).sort_values(by='gwp',ascending=False)
print(df_test)
print('')

# uuids = ['ec329xb7', 'ec3mfenf']
uuids = df_test.index

#in bulk
# cat = 'description'
# for uuid in uuids:
#     epd = ec3_epds.get_epd_by_xpduuid(uuid)
#     print(epd[cat])
#     print('')


#one off
# uuid = uuids[0]
# epd = ec3_epds.get_epd_by_xpduuid(uuid)
# pprint(epd)


#look for phrase in category
# cat = 'description'
# phrase = 'securock'
# count = 0
# uuids = []
# for epd in epd_dict_filtered[mat]:
#     if cat in epd:
#         if phrase in epd[cat].lower():
#             uuid = epd['open_xpd_uuid']
#             uuids.append(uuid)
#             count += 1
            # print(epd['description'])
            # print(epd['declared_unit'])
            # print(epd['open_xpd_uuid'])
            # print('')
# print(count)

           gwp
ec3a65uz  62.7
ec3mua8g  46.3
ec33a9ys  30.6



In [244]:
# mat = 'gypsum_kg'
# df_test = pd.DataFrame({
#     'gwp': gwp_dict[mat]['gwp'],
#     'uuid': gwp_dict[mat]['uuids'],
# })

# df_test.loc[3,'uuid']
# for row in range(len(df_test)):
#     if df_test.loc[row,'uuid'] in uuids:
#         print(df_test.loc[row,'gwp'])

# Materials not in EC3

In [56]:
%%time
# MATERIALS NOT IN EC3
nonec3_materials = {
    # 'stone_m3': {
    #     'gwp_list': [(6.85e0+1.65e0+1.29e1)/0.05], #A1-A3, per m3
    #     'declared_unit': 'm3',
    # },
    # https://www.naturalstoneinstitute.org/programs/sustainability/sustainability-resources/
    # https://transparencycatalog.com/assets/uploads/pdf/Exterior-Dimension-Stone-Cladding_NSI.pdf
    # components of stone cladding (functional unit 1 m2 for 75 years)

    # thickness = 0.05m
    # natural stone - 83.28 kg/m2
    # mortar - 4.88 kg/m2
    # masonry connectors - 0.62 kg/m2
    # water - 1.00 L/m2 = 1 kg/m2
    
    'glulam_m3': {
        'gwp_list': [
            197.97, #https://www.cwc.ca/wp-content/uploads/2019/03/EPD-Glulam-1.pdf
            124.50, #https://kalesnikoff.com/wp-content/uploads/2022/05/Kalesnikoff-Glulam-EPD-.pdf
            122.0, #Element5 LP - Modern Timber Buildings
            -6.60e2+6.26e0+1.44e1, #https://www.binderholz.com/fileadmin/user_upload/pdf/approvals-certifications/EPD-binderholz-Brettschichtholz-BSH-ENG---28-11-2024.pdf
            -6.08e2, #https://www.hasslacher.com/data/_dateimanager/zertifikate/en/Glued_laminated_timber/EPD_GLT_GST_BGG_SC_ISO_14025_EN_15804_A2_HASSLACHER_Holding_GmbH_en.pdf
            147.99, #https://pcr-epd.s3.us-east-2.amazonaws.com/644.EPD_FOR_Vaagen_Timbers__Gulam_EPD_20210414.pdf
        ],
        'declared_unit': 'm3',
    },
    
}

for mat in nonec3_materials:
    gwp_dict[mat] = dict.fromkeys(categories)
    gwp_dict[mat]['gwp'] = nonec3_materials[mat]['gwp_list']
    gwp_dict[mat]['declared_unit'] = nonec3_materials[mat]['declared_unit']

CPU times: user 13 μs, sys: 1e+03 ns, total: 14 μs
Wall time: 16 μs


# Get density per m3 and m2

In [57]:
#find density for EC3 materials
for mat in epd_dict_filtered:
    densities = return_densities(epd_dict_filtered[mat])
    densities = [ele for ele in densities if type(ele) is float or type(ele) is int]
    if len(densities) > 0:
        gwp_dict[add_material_unit_dict[mat]]['density_kgm3'] = np.median(densities)
        

In [58]:
#find density for non-EC3 materials or if it wasn't in EC3

#shingles
gwp_dict['shingles_kg']['density_kgm3'] = shingle_kgm2/(6.1/1_000)

# glulam
# https://harvest-timber.com/wp-content/uploads/2013/12/Glulam-Design-Properties-and-Layup-Combinations.pdf
gwp_dict['glulam_m3']['density_kgm3'] = density2kgm3(35, 'lb/ft3')

# clay
# https://www.engineeringtoolbox.com/ceramics-properties-d_1227.html#gsc.tab=0
gwp_dict['clay_m2']['density_kgm3'] = 4500

#tile
for mat in gwp_dict.keys():
    if mat.find('tile') != -1:
        gwp_dict[mat]['density_kgm3'] = 4500
    

#gypsum
#https://www.aqua-calc.com/page/density-table/substance/gypsum-coma-and-blank-solid
# gwp_dict['gypsum_m2']['density_kgm3'] = 2960 #kg/m3

gwp_dict = dict(sorted(gwp_dict.items())) #sort dictionary by keys


In [59]:
unaccounted = []
for mat in epd_dict_filtered:
    density = gwp_dict[add_material_unit_dict[mat]]['density_kgm3']
    if density == None:
        unaccounted.append(mat)

if len(unaccounted) > 0:
    raise ValueError(f'{unaccounted} do not yet have densities')
    

In [60]:
### ADD kg/m2 DENSITY FOR M2
#get_kg_m2rsi(fgbatt_records)

for mat in gwp_dict:
    if gwp_dict[mat]['declared_unit'] == 'm2':
        gwp_dict[mat]['density_kgm2'] = np.median(get_kg_m2(epd_dict_filtered[mat[:mat.find('_')]], prnt='n'))
        
# epd_dict_filtered['gypsum_m2']['density_kgm2'] = 2960*0.0127 #kg/m2 for 1/2" thick



/Users/martintorres/mambaforge/envs/waterweed/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/martintorres/mambaforge/envs/waterweed/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


# Read kde_dict from JSON, update material with df_material_unit, write kde_dict to JSON
# _
# _
# _

In [61]:
%%time
label_dict = {
    'stone_m3': 'Stone (m3)',
    'concrete3ksi_m3': f'Concrete (m3)',
    'concrete4ksi_m3': f'Concrete, 4ksi (m3)',
    'concreteTX3ksi_m3': f'Concrete, TX (m3)',
    'concreteTX4ksi_m3': f'Concrete, 4ksi, TX (m3)',
    'concreteTX_m3': f'Concrete, TX (m3)',
    'concrete_m3': f'Concrete (m3)',
    'dimlumber_m3': f'Dim Lumber (m3)',
    'fgbatt_m2': f'Fiberglass Batt (m2rsi)',
    'glass_kg': f'Glass (kg)',
    'glass_m2': 'Glass (m2)',
    'gypsum_kg': 'Gypsum Board (kg)',
    'gypsum_m2': f'Gypsum Board (m2)',
    'gypsumlight_kg': 'Gypsum Board, Light (kg)',
    'plywood_m3': f'Plywood (m3)',
    'roofmembranes_m2': f'Roof Membranes (m2)',
    'shingles_kg': 'Shingles (kg)',
    'steelrebar_kg': f'Steel, Rebar (kg)',
    'stone_m3': f'Stone (m3)',
    'wood_m3': f'Wood Products (m3)',
    'xps_m2': f'XPS (m2rsi)',
    
    
    ##########################################
    # NON DOE MATERIALS
    ##########################################
    'blanket_m2': 'Blanket Insulation, (m2)',
    'brick_kg': 'Brick (kg)',
    'carpet_m2': 'Carpet (m2)',
    'cellulose_m2': 'Cellulose (m2rsi)',
    'clay_m2': 'Clay (m2)',
    'clt_m3': 'CLT (m3)',
    'eps_m2': 'EPS (m2rsi)',
    'glulam_kg': 'Glulam (kg)',
    'glulam_m3': 'Glulam (m3)',
    'gypsumlight_m2': 'Gypsum Board, Light (m2)',
    'hardieboard_m2': 'Hardieboard (m2)',
    'insulation_m2': 'Insulation (m2)',
    'metalsheet_kg': 'Metal Sheet (kg)',
    'mineralwool_m2': 'Mineral Wool (m2rsi)',
    'osb_m2': 'OSB (m2)',
    'osb_m3': 'OSB (m3)',
    'rockwool_m2': 'Rockwool (m2rsi)',
    'siding_m2': 'Siding (m2)',
    'sprayfoam_m2': 'Sprayfoam (m2rsi)',
    'steelcoldformed_kg': 'Steel, Cold-Formed (kg)',
    'steeldeck_kg': 'Steel Deck (kg)',
    'steelstructural_kg': 'Steel, Structural (kg)',
    'tile_m2': 'Tile (m2)',
    'tile_kg': 'Tile (kg)',
    'tile_m3': 'Tile (m3)',
}

not_included = []
for mat in gwp_dict:
    if mat not in label_dict:
        not_included.append(mat)

if len(not_included) > 0:
    raise ValueError(f'{not_included} are not in label_dict')
    
for mat in gwp_dict:
    gwp_dict[mat]['label'] = label_dict[mat]
    gwp_dict[mat]['label_noUnit'] = label_dict[mat].split('(')[0][:-1]

CPU times: user 48 μs, sys: 1 μs, total: 49 μs
Wall time: 52 μs


In [62]:
%%time
mats = gwp_dict.keys()

#check that all materials are represented in label_dict
for mat_unit in mats:
    if mat_unit not in label_dict:
        raise ValueError(f'{mat_unit} is not represented in label_dict')

#read json file
with open('kde_dict.json', 'r') as read_file:
    kde_dict = json.load(read_file)

#make changes
mats2del = []
for mat_unit in mats:
    mat = mat_unit[:mat_unit.find('_')]
    #delete if there are no usable values
    if len(gwp_dict[mat_unit]['gwp']) == 0:
        mats2del.append(mat_unit)
        print(mat_unit)
        continue
    
    kde_dict[mat_unit] = dict.fromkeys(categories)
    kde_dict[mat_unit]['gwp'] = gwp_dict[mat_unit]['gwp']
    kde_dict[mat_unit]['uuids'] = gwp_dict[mat_unit]['uuids']
    kde_dict[mat_unit]['declared_unit'] = gwp_dict[mat_unit]['declared_unit']
    kde_dict[mat_unit]['density_kgm3'] = gwp_dict[mat_unit]['density_kgm3']
    kde_dict[mat_unit]['date_pulled'] = datetime.now().strftime('%Y-%m-%d')
    
    
    if mat in epd_dict_filtered:
        n_EPD_i = len(epd_dict_filtered[mat])
    else:
        n_EPD_i = len(gwp_dict[mat_unit]['gwp'])
    kde_dict[mat_unit]['n_EPD_i'] = n_EPD_i
    kde_dict[mat_unit]['n_EPD_f'] = len(gwp_dict[mat_unit]['gwp'])
    kde_dict[mat_unit]['label'] = label_dict[mat_unit]
    kde_dict[mat_unit]['label_noUnit'] = label_dict[mat_unit].split('(')[0][:-1]

    if gwp_dict[mat_unit]['declared_unit'] == 'm2':
        kde_dict[mat_unit]['density_kgm2'] = gwp_dict[mat_unit]['density_kgm2']


#delete materials with no usable values
for mat_unit in mats2del:
    if mat_unit in kde_dict:
        del kde_dict[mat_unit]
    if mat_unit in epd_dict:
        del epd_dict[mat_unit]
    if mat_unit in epd_dict_filtered:
        del epd_dict_filtered[mat_unit]
#sort dictionary
kde_dict = dict(sorted(kde_dict.items()))
        
#write json file with changes
json_object = json.dumps(kde_dict, indent=4)
with open('kde_dict.json', 'w') as write_file:
    write_file.write(json_object)

clay_m2
tile_m3
CPU times: user 122 ms, sys: 14.6 ms, total: 137 ms
Wall time: 140 ms
